# simulate_arch

> 对应代码：`EconometricsCode/code_in_notes/Chap.28/simulate_arch.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.28`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.28")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

设置 1000 期观测与随机种子，保证模拟结果可复现。

In [ ]:
%%stata
clear
set obs 1000
set more off
set seed 505

设定 ARCH(2) 的参数：常数项 α₀=1，滞后一阶与二阶系数 α₁=α₂=0.3。注意 α₁+α₂=0.6<1，满足协方差平稳条件。

In [ ]:
%%stata
// 设定参数
local alpha0=1
local alpha1=0.3
local alpha2=0.3

生成时间索引并声明时间序列；z 为标准正态白噪声，是 ARCH 过程中的创新（innovation）。

In [ ]:
%%stata
// 开始模拟
gen t=_n
tsset t
gen z=rnormal()

递推生成 ARCH(2) 过程：前两期 ε 取 0 作为初始值，从第 3 期起 ε_t = √(α₀+α₁ε²_{t-1}+α₂ε²_{t-2})·z_t。条件标准差由滞后残差平方决定，从而波动具有时变性与聚集性，但无条件分布仍是对称的厚尾分布。

In [ ]:
%%stata
gen epsilon=0 if t<=2
replace epsilon=sqrt(`alpha0'+`alpha1'*L.epsilon^2+`alpha2'*L2.epsilon^2)*z if t>2

画出 ε 的时序图，可直观看到大幅波动之后往往跟随大幅波动的聚集现象。

In [ ]:
%%stata
// 画图
tsline epsilon

计算 ε² 的自相关函数（ACF）。ARCH 过程本身不相关（均值零、序列不相关），但其平方序列显著正自相关，这正是条件异方差的诊断特征。

In [ ]:
%%stata
gen epsilon2=epsilon^2
ac epsilon2

输出 ε 的详细描述性统计（含偏度、峰度），用以验证 ARCH 过程对称但厚尾（峰度大于 3）的特征事实。

In [ ]:
%%stata
su epsilon, de